# SmartWalker YOLO 커스텀 학습

시각장애인 보행 보조용 객체 인식 모델 학습 노트북입니다.

**목표 클래스 (6개)**

| ID | 클래스명 | 설명 |
|----|----------|------|
| 0 | `traffic_light_red` | 적색 신호등 |
| 1 | `traffic_light_green` | 녹색 신호등 |
| 2 | `crosswalk` | 횡단보도 |
| 3 | `obstacle` | 보행 장애물 |
| 4 | `stairs` | 계단 |
| 5 | `tactile_paving` | 점자블록 |

**시작 전 체크**
- 런타임 → 런타임 유형 변경 → **T4 GPU** 선택
- Roboflow 계정의 Private API Key 준비 (roboflow.com → 프로필 → API)

## 셀 1 — 패키지 설치 및 Drive 마운트

In [ ]:
!pip install -q ultralytics roboflow fiftyone

from google.colab import drive
drive.mount('/content/drive')

import os, shutil, yaml, random
from pathlib import Path
from collections import defaultdict, Counter

DRIVE_DIR = "/content/drive/MyDrive/smartwalker_yolo"
os.makedirs(DRIVE_DIR, exist_ok=True)
print("준비 완료")

## 셀 2 — 설정

> **여기서만 수정합니다.** API 키를 입력하세요.

In [ ]:
# Roboflow Private API Key
# roboflow.com → 우상단 프로필 → Roboflow API → copy
ROBOFLOW_API_KEY = "여기에_API_키_입력"

# 우리 클래스 정의
CLASS_NAMES = [
    "traffic_light_red",    # 0
    "traffic_light_green",  # 1
    "crosswalk",            # 2
    "obstacle",             # 3
    "stairs",               # 4
    "tactile_paving",       # 5
]
NC = len(CLASS_NAMES)

# 소스 데이터셋 클래스명 → 우리 ID 매핑
CLASS_REMAP = {
    # 신호등
    "red": 0, "red_light": 0, "traffic_light_red": 0,
    "stop": 0, "red light": 0,
    "green": 1, "green_light": 1, "traffic_light_green": 1,
    "go": 1, "green light": 1,
    # 횡단보도
    "crosswalk": 2, "crosswalk_line": 2, "zebra_crossing": 2,
    "pedestrian crossing": 2, "zebra crossing": 2,
    # 계단
    "stairs": 4, "staircase": 4, "steps": 4,
    # 보행 장애물 → obstacle (3)
    "person": 3, "pedestrian": 3,
    "car": 3, "vehicle": 3, "automobile": 3,
    "bicycle": 3, "bike": 3,
    "motorcycle": 3, "motorbike": 3,
    "bus": 3, "truck": 3,
    "scooter": 3, "bollard": 3,
    # 점자블록 (tactile-paving-detection-analz 의 3개 클래스 포함)
    "tactile_paving": 5, "braille_block": 5, "braille block": 5,
    "tactile": 5, "braille": 5,
    "horizontal-directional-tile": 5,
    "vertical-directional-tile": 5,
    "warning-tile": 5,
    "directional tile": 5, "warning tile": 5,
}

DATASET_ROOT = Path("/content/dataset")
MERGED_DIR   = Path("/content/dataset/merged")
print("설정 완료")

## 셀 3 — Roboflow 데이터셋 다운로드

| 데이터셋 | 이미지 수 | 커버 클래스 |
|----------|-----------|-------------|
| Road Signs & Traffic Lights (color) | 2,133 | 신호등(적/녹) |
| Pedestrian Traffic Light | 926 | 신호등(적/녹) |
| YOLOv8 Crosswalk Detection | 1,049 | 횡단보도 |
| Braille Blocks Detection | 638 | 점자블록 |
| Braille Block (large) | 1,872 | 점자블록 |
| Tactile Paving Detection | 206 | 점자블록 |

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY)

def download_latest(workspace: str, project_name: str, dest: str) -> str:
    """최신 버전을 YOLOv8 포맷으로 다운로드하고 경로를 반환."""
    project = rf.workspace(workspace).project(project_name)
    versions = project.versions()
    if not versions:
        raise RuntimeError(f"버전 없음: {workspace}/{project_name}")
    latest_ver = max(v.version for v in versions)
    print(f"  {project_name}: v{latest_ver} 다운로드 중...")
    ds = project.version(latest_ver).download("yolov8", location=dest)
    img_count = len(list(Path(dest).rglob("*.jpg"))) + len(list(Path(dest).rglob("*.png")))
    print(f"  → {dest} ({img_count}장)")
    return dest

ROBOFLOW_DATASETS = [
    # (workspace, project, 로컬 저장 폴더명)
    (
        "internship-tvkfo",
        "road-signs-and-traffic-lights-detection-and-color-recognition-using-yolov8",
        "traffic_light_color",
    ),
    (
        "ono-gedd7",
        "pedestrian-traffic-light-puf4a",
        "pedestrian_traffic_light",
    ),
    (
        "tcc-xn3st",
        "yolov8-crosswalk-detection",
        "crosswalk",
    ),
    (
        "tong-gae",
        "braille-blocks-detection",
        "braille_blocks",
    ),
    (
        "braille-block-qdtxl",
        "braille-block-f7vqq",
        "braille_block_large",
    ),
    (
        "ods-8nj8c",
        "tactile-paving-detection-analz",
        "tactile_paving",
    ),
]

roboflow_dirs = []
for workspace, project, folder in ROBOFLOW_DATASETS:
    dest = str(DATASET_ROOT / "roboflow" / folder)
    try:
        download_latest(workspace, project, dest)
        roboflow_dirs.append(dest)
    except Exception as e:
        print(f"  ⚠ {project} 실패: {e}")

print(f"\n다운로드 완료: {len(roboflow_dirs)}개")

## 셀 4 — Open Images v7 다운로드 (계단 보강)

계단(`stairs`)은 Roboflow 데이터셋에 없으므로 Open Images에서 보강합니다.

In [ ]:
import fiftyone as fo
import fiftyone.zoo as foz

def load_oi(dataset_name, classes, max_samples, export_dir, id_remap):
    """Open Images v7 다운로드 -> id_remap 적용 -> export_dir 저장."""
    if fo.dataset_exists(dataset_name):
        fo.delete_dataset(dataset_name)

    ds = foz.load_zoo_dataset(
        "open-images-v7",
        split="train",
        label_types=["detections"],
        classes=classes,
        max_samples=max_samples,
        seed=42,
        dataset_name=dataset_name,
    )

    # FiftyOne 버전마다 필드명이 다를 수 있으므로 동적 탐색
    schema = ds.get_field_schema()
    det_field = "detections"
    if "detections" not in schema:
        for fname, ftype in schema.items():
            if "Detection" in str(type(ftype)):
                det_field = fname
                break
        print(f"⚠ {dataset_name}: det_field={det_field}")

    view = ds.exists(det_field)
    print(f"{dataset_name}: {len(view)}/{len(ds)} 유효 샘플")

    view.export(
        export_dir=export_dir,
        dataset_type=fo.types.YOLOv5Dataset,
        label_field=det_field,
        classes=classes,
    )

    for split in ["train", "val"]:
        lbl_dir = Path(export_dir) / split / "labels"
        if not lbl_dir.exists():
            continue
        for txt_file in lbl_dir.glob("*.txt"):
            lines_out = []
            for line in txt_file.read_text().strip().splitlines():
                parts = line.split()
                src_id = int(parts[0])
                if src_id in id_remap:
                    lines_out.append(f"{id_remap[src_id]} {' '.join(parts[1:])}")
            txt_file.write_text("\n".join(lines_out))

    return export_dir


# ── 1. 계단 (class 4) ────────────────────────────────────────────
STAIRS_CLASSES = ["Stairs"]
OI_STAIRS_DIR = str(DATASET_ROOT / "oi_stairs")
load_oi(
    "oi_stairs",
    STAIRS_CLASSES,
    max_samples=400,
    export_dir=OI_STAIRS_DIR,
    id_remap={0: 4},
)

# ── 2. 보행 장애물 (class 3) ─────────────────────────────────────
# Person/Car/Bicycle 등 보행로 장애물이 될 수 있는 클래스를 모두 3으로 매핑
OBS_CLASSES = ["Person", "Car", "Bicycle", "Motorcycle", "Bus", "Truck"]
OI_OBS_DIR = str(DATASET_ROOT / "oi_obstacle")
load_oi(
    "oi_obstacle",
    OBS_CLASSES,
    max_samples=800,
    export_dir=OI_OBS_DIR,
    id_remap={i: 3 for i in range(len(OBS_CLASSES))},
)

OI_DIRS = [OI_STAIRS_DIR, OI_OBS_DIR]
print("Open Images 완료")

## 셀 5 — Roboflow 클래스 재매핑

각 데이터셋의 클래스명을 `CLASS_REMAP`에 따라 우리 ID로 통일합니다.

In [ ]:
def load_yaml_names(dataset_dir: str) -> list:
    for yaml_name in ["data.yaml", "dataset.yaml", "_annotations.yaml"]:
        p = Path(dataset_dir) / yaml_name
        if p.exists():
            with open(p) as f:
                meta = yaml.safe_load(f)
            names = meta.get("names", [])
            return names if isinstance(names, list) else list(names.values())
    return []

def remap_roboflow(dataset_dir: str):
    src_names = load_yaml_names(dataset_dir)
    if not src_names:
        print(f"  ⚠ YAML 없음: {dataset_dir}")
        return

    id_remap = {}
    for src_id, name in enumerate(src_names):
        key = name.lower().strip()
        if key in CLASS_REMAP:
            id_remap[src_id] = CLASS_REMAP[key]
        else:
            for pattern, our_id in CLASS_REMAP.items():
                if pattern in key or key in pattern:
                    id_remap[src_id] = our_id
                    break

    mapping_display = {src_names[i]: id_remap.get(i, "skip") for i in range(len(src_names))}
    print(f"  {Path(dataset_dir).name}: {mapping_display}")

    for split in ["train", "valid", "val", "test"]:
        lbl_dir = Path(dataset_dir) / split / "labels"
        if not lbl_dir.exists():
            continue
        for txt in lbl_dir.glob("*.txt"):
            new_lines = []
            for line in txt.read_text().strip().splitlines():
                parts = line.split()
                if not parts:
                    continue
                src_id = int(parts[0])
                if src_id in id_remap:
                    new_lines.append(f"{id_remap[src_id]} {' '.join(parts[1:])}")
            txt.write_text("\n".join(new_lines))

print("클래스 재매핑 시작...")
for d in roboflow_dirs:
    remap_roboflow(d)
print("완료")

## 셀 6 — 데이터셋 병합 및 분할 (8 : 1.5 : 0.5)

In [ ]:
ALL_SOURCES = roboflow_dirs + OI_DIRS

def collect_pairs(src_dir: str) -> list:
    """(이미지 경로, 라벨 경로) 유효한 쌍만 반환."""
    pairs = []
    for split in ["train", "valid", "val", "test"]:
        img_dir = Path(src_dir) / split / "images"
        lbl_dir = Path(src_dir) / split / "labels"
        if not img_dir.exists():
            continue
        for img in sorted(img_dir.glob("*.[jp][pn]g")):
            lbl = lbl_dir / img.with_suffix(".txt").name
            if lbl.exists() and lbl.stat().st_size > 0:
                pairs.append((img, lbl))
    return pairs

all_pairs = []
for src in ALL_SOURCES:
    pairs = collect_pairs(src)
    print(f"  {Path(src).name}: {len(pairs)}쌍")
    all_pairs.extend(pairs)

print(f"\n전체: {len(all_pairs)}쌍")

random.seed(42)
random.shuffle(all_pairs)
n       = len(all_pairs)
n_train = int(n * 0.80)
n_val   = int(n * 0.15)

split_assignments = (
    [(p, "train") for p in all_pairs[:n_train]]
    + [(p, "val")   for p in all_pairs[n_train:n_train + n_val]]
    + [(p, "test")  for p in all_pairs[n_train + n_val:]]
)

for split in ["train", "val", "test"]:
    (MERGED_DIR / split / "images").mkdir(parents=True, exist_ok=True)
    (MERGED_DIR / split / "labels").mkdir(parents=True, exist_ok=True)

for (img, lbl), split in split_assignments:
    # 파일명 충돌 방지: 소스 폴더명 prefix 추가
    parts = img.parts
    prefix = parts[-4] if len(parts) >= 4 else "src"
    dst_name = f"{prefix}_{img.name}"
    shutil.copy(img, MERGED_DIR / split / "images" / dst_name)
    shutil.copy(lbl, MERGED_DIR / split / "labels" / (Path(dst_name).stem + ".txt"))

print(f"병합 완료 — train:{n_train} / val:{n_val} / test:{n - n_train - n_val}")

data_yaml = {
    "path":  str(MERGED_DIR),
    "train": "train/images",
    "val":   "val/images",
    "test":  "test/images",
    "nc":    NC,
    "names": CLASS_NAMES,
}
with open(MERGED_DIR / "data.yaml", "w") as f:
    yaml.dump(data_yaml, f, allow_unicode=True, default_flow_style=False)

print("data.yaml 생성 완료")

## 셀 7 — 클래스 분포 확인

200개 미만(△) 클래스는 셀 3의 `ROBOFLOW_DATASETS`에 추가 데이터셋을 넣고 재실행하세요.

In [ ]:
counter = Counter()
for txt in (MERGED_DIR / "train" / "labels").glob("*.txt"):
    for line in txt.read_text().strip().splitlines():
        parts = line.split()
        if parts:
            counter[int(parts[0])] += 1

print(f"{'ID':<4} {'클래스':<25} {'개수':>6}  상태")
print("-" * 48)
for cls_id, name in enumerate(CLASS_NAMES):
    count  = counter.get(cls_id, 0)
    status = "✓" if count >= 200 else ("△ 부족" if count >= 50 else "✗ 매우 부족")
    print(f"{cls_id:<4} {name:<25} {count:>6}  {status}")

## 셀 8 — 학습

학습 결과는 Google Drive `smartwalker_yolo/runs/` 에 저장되므로 세션이 끊겨도 유지됩니다.

In [ ]:
import os
from ultralytics import YOLO

last_pt = f"{DRIVE_DIR}/runs/smartwalker_v1/weights/last.pt"

if os.path.exists(last_pt):
    print(f"체크포인트 발견 — 이어서 학습: {last_pt}")
    model = YOLO(last_pt)
    results = model.train(resume=True)
else:
    print("체크포인트 없음 — 처음부터 학습 시작")
    model = YOLO("yolov8n.pt")   # nano — 모바일 최적화
    results = model.train(
        data     = str(MERGED_DIR / "data.yaml"),
        epochs   = 100,
        imgsz    = 640,
        batch    = 16,           # OOM 시 8로 줄임
        name     = "smartwalker_v1",
        patience = 20,
        augment  = True,
        hsv_h    = 0.01,         # 신호등 색상 오인식 방지 — 색조 augment 최소화
        hsv_s    = 0.5,
        hsv_v    = 0.4,
        project  = f"{DRIVE_DIR}/runs",
    )

BEST_PT = f"{DRIVE_DIR}/runs/smartwalker_v1/weights/best.pt"
print(f"학습 완료: {BEST_PT}")

## 셀 9 — 성능 확인

mAP50 기준: ✓ ≥ 0.65 / △ 0.40–0.64 / ✗ < 0.40 (재학습 필요)

In [ ]:
# 세션 재시작 후 단독 실행 가능
from ultralytics import YOLO
from pathlib import Path
import shutil

DRIVE_DIR  = "/content/drive/MyDrive/smartwalker_yolo"
MERGED_DIR = Path("/content/dataset/merged")
BEST_PT    = f"{DRIVE_DIR}/runs/smartwalker_v1/weights/best.pt"
CLASS_NAMES = [
    "traffic_light_red", "traffic_light_green", "crosswalk",
    "obstacle", "stairs", "tactile_paving",
]

model   = YOLO(BEST_PT)
metrics = model.val(data=str(MERGED_DIR / "data.yaml"))

sep = "-" * 58
hdr = "{:<4} {:<25} {:>7} {:>10}  상태".format("ID", "클래스", "mAP50", "mAP50-95")
print(sep)
print(hdr)
print(sep)
for i, name in enumerate(CLASS_NAMES):
    ap50 = float(metrics.box.ap50[i]) if i < len(metrics.box.ap50) else 0.0
    ap   = float(metrics.box.ap[i])   if i < len(metrics.box.ap)   else 0.0
    flag = "OK" if ap50 >= 0.65 else ("low" if ap50 >= 0.40 else "FAIL")
    print("{:<4} {:<25} {:>7.3f} {:>10.3f}  {}".format(i, name, ap50, ap, flag))

print(sep)
print("전체 mAP50: {:.3f}  (목표 >= 0.65)".format(metrics.box.map50))


## 셀 10 — TFLite 변환 및 Drive 저장

INT8 양자화: 모델 크기 ~4 MB, Pixel 6 기준 추론 ~30 ms  
INT8 후 mAP가 5% 이상 떨어지면 마지막 셀의 FP16 변환을 사용하세요.

In [ ]:
# 세션 재시작 후 단독 실행 가능
from ultralytics import YOLO
from pathlib import Path
import shutil, glob

DRIVE_DIR  = "/content/drive/MyDrive/smartwalker_yolo"
MERGED_DIR = Path("/content/dataset/merged")
BEST_PT    = f"{DRIVE_DIR}/runs/smartwalker_v1/weights/best.pt"
CLASS_NAMES = [
    "traffic_light_red", "traffic_light_green", "crosswalk",
    "obstacle", "stairs", "tactile_paving",
]

model = YOLO(BEST_PT)

model.export(
    format = "tflite",
    imgsz  = 640,
    int8   = True,
    data   = str(MERGED_DIR / "data.yaml"),
)

tflite_files = glob.glob(
    str(Path(BEST_PT).parent.parent / "**/*.tflite"),
    recursive=True,
)
for f in tflite_files:
    dst = f"{DRIVE_DIR}/{Path(f).name}"
    shutil.copy(f, dst)
    print("저장됨: " + dst)

labels_path = f"{DRIVE_DIR}/labels.txt"
Path(labels_path).write_text(chr(10).join(CLASS_NAMES))
print("저장됨: " + labels_path)
print("완료! Drive에서 두 파일을 다운로드하세요.")


## (선택) FP16 변환 — INT8 정확도 저하 시 대안

In [ ]:
model = YOLO(BEST_PT)
model.export(format="tflite", imgsz=640, int8=False, half=True)

fp16_files = _glob.glob(
    str(Path(BEST_PT).parent.parent / "**/*.tflite"),
    recursive=True,
)
for f in fp16_files:
    dst = f"{DRIVE_DIR}/fp16_{Path(f).name}"
    shutil.copy(f, dst)
    print(f"저장됨: {dst}")

## Android 앱 적용

Drive에서 `.tflite`와 `labels.txt`를 로컬로 다운로드한 뒤:

```
android/app/src/main/assets/yolov8n.tflite   ← tflite 파일로 교체
android/app/src/main/assets/labels.txt       ← labels.txt 교체
```

`DetectionLabels.kt`에서 위험 클래스 목록 확인:
```kotlin
val DANGER_CLASSES = setOf("obstacle", "stairs", "traffic_light_red")
```